# Track A — Bước A1c: Kết hợp Toàn ảnh + Auto-ROI (`--roi_fusion`) trên `GC10-DET`
- Nhánh: `duytung/03-gc10-dac-trung-vung-loi`
- Cơ chế: Fine-tune `DINO ViT-B/16` trên **ảnh toàn phần** (`FT_EPOCHS=20`, giữ vững `Old`), sau đó trích và hợp nhất L2 đặc trưng **`Full + Auto-ROI`** (`--roi_fusion`) rồi chạy `r0`, `s0..s4`.
- Tự động đo **CPU % peak/avg, RAM peak GB, GPU % peak, GPU VRAM peak MB** cho **từng công đoạn** và toàn bộ quá trình, ghi vào `gc10_a1c_summary.txt` / `gc10_a1c_summary.json`.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo lên ổ local /content (chạy git trong 2 giây)
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b duytung/03-gc10-dac-trung-vung-loi https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin duytung/03-gc10-dac-trung-vung-loi
!git checkout duytung/03-gc10-dac-trung-vung-loi
!git reset --hard origin/duytung/03-gc10-dac-trung-vung-loi
!git status -sb

In [ ]:
# 3. Cài gói thiếu, đảm bảo JAX nhận GPU, và kiểm tra GC10-DET trên Drive
!pip uninstall -y -q torchao
!pip install -q continuum numpyro prettytable

import os, subprocess
has_gpu = subprocess.call(
    ['python', '-c', 'import jax; assert any(d.platform != "cpu" for d in jax.devices())']
) == 0
if not has_gpu:
    print('>>> JAX đang chạy CPU, thử cài jax[cuda12]...')
    !pip install -q -U "jax[cuda12]"

!python -c "import torch, jax, numpyro, continuum, prettytable, sklearn, scipy, datasets, peft; print('Imports OK! Torch CUDA:', torch.cuda.is_available(), '| JAX devices:', jax.devices())"

cands = ['/content/drive/MyDrive/Datasets/GC10-DET', '/content/drive/MyDrive/VB_CGCD_Work/GC10-DET', '/content/drive/MyDrive/VB_CGCD_Work/Datasets/GC10-DET']
found = [p for p in cands if os.path.isfile(f'{p}/1/img_06_425613700_00781.jpg')]
if found:
    print('>>> GC10-DET sẵn sàng tại:', found[0])
else:
    raise FileNotFoundError('Không tìm thấy GC10-DET trên Drive.')

In [ ]:
# 4. Chạy ngầm Bước A1c (STAGE='auto' | 'eval')
STAGE = 'auto'

import os, subprocess, time

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a1c_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a1c_bg.pid'

running = False
if os.path.exists(PID_FILE):
    old_pid = open(PID_FILE).read().strip()
    if old_pid and os.path.exists(f'/proc/{old_pid}/status'):
        st = open(f'/proc/{old_pid}/status').read()
        if 'State:\tZ' not in st and 'State: Z' not in st:
            running = True
            print(f'>>> Pipeline đang chạy ngầm sẵn với PID {old_pid}. Chạy Cell 5 để xem log!')

if not running:
    proc = subprocess.Popen(
        ['nohup', 'bash', 'colab/run_gc10_a1c_bg.sh', STAGE],
        stdout=open(LOG_FILE, 'w'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    time.sleep(2)
    print(f'>>> Đã khởi chạy ngầm STAGE={STAGE}, PID={proc.pid}! Log: {LOG_FILE}')
    !head -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ, CPU/RAM/GPU peak từng giai đoạn & bảng kết quả Bước A1c
import os

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a1c_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a1c_bg.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a1c_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = 'State:\tZ' not in st and 'State: Z' not in st
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

if os.path.exists(LOG_FILE):
    print('\n--- 40 dòng log gần nhất ---')
    !tail -n 40 {LOG_FILE}

if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A1c ---')
    print(open(SUMMARY_OUT).read())